# Scheduling a conference with clashless

`clashless` assigns conference presentations to `(day, session)` slots so that:

- nobody is double-booked, whether they're a **student**, a **supervisor** (`s1_name`/`s2_name`),
  or a **moderator** — a moderator can also be one of a presentation's own supervisors;
- nobody is scheduled during a time they've marked themselves **unavailable**;
- and any number of presentations can run **in parallel**, since a "room" isn't a separate
  resource — it's simply defined by *who is moderating*. Two presentations can share a
  `(day, session)` slot as long as they don't share any of the same people.

This tutorial works through a single realistic-sized example: the ~290-presentation synthetic
dataset shipped in `tests/data/large_synthetic` (generated by `tests/datamaker.ipynb`), which has
6 moderators, 8 sessions per day, and 60 unavailability rules spread across the supervisor pool.

## Loading the data

`clashless` has one class per input table: `Presentations`, `Unavailability`, and `SessionTimes`.
Each one just loads and validates its CSV, exposing the parsed table as `.data`.

We point them at the fixture directory directly (this notebook lives in `docs/tutorials/`, and the
data lives in `tests/data/large_synthetic/`, two directories up from the repo root).

In [1]:
import pathlib

from clashless import Presentations, Schedule, SchedulingError, SessionTimes, Unavailability

data_dir = pathlib.Path("../../tests/data/large_synthetic")

presentations = Presentations(data_dir / "presentations.csv")
unavailability = Unavailability(data_dir / "unavailable.csv")
session_times = SessionTimes(data_dir / "session-start-times.csv")

presentations.data.head()

,student,s1_name,s2_name,moderator
id,,,,
tby46,Tao Yi,Catherine Brown,Justin Gomez,Kelly Garner
fql33,Fang Lai,Mitchell Bailey,Darlene Schwartz,James Lowery
dt5008,Derek Tucker,Brandon Sanchez,Linda Robertson,Amanda Castro
hkc953,Heather Campbell,Edward Flores,Justin Allen,Amanda Ramos
cy306,Chao Yin,Justin Allen,Monique Cole,Luis Reynolds


## Getting a feel for the data

How many presentations are there, and how is the moderator workload distributed? Since a
moderator effectively *is* a room for the whole day they're chairing, the moderator with the most
presentations is the tightest constraint on how many days the conference needs.

In [2]:
print(f"{len(presentations.data)} presentations")
print(f"{session_times.n_sessions} sessions per day")
print(f"{len(unavailability.data)} unavailability rules")

moderator_load = presentations.data["moderator"].value_counts()
moderator_load

290 presentations
8 sessions per day
60 unavailability rules


moderator
Kelly Garner     53
Luis Reynolds    52
Amanda Castro    51
Cindy James      47
Amanda Ramos     44
James Lowery     43
Name: count, dtype: int64

## Trying an infeasible conference length

The busiest moderator chairs over 50 presentations, so with 8 sessions per day, a handful of days
isn't enough — that moderator alone would need more distinct slots than exist. `clashless` doesn't
try to tell you how many days you need; `Schedule` is **feasibility-only** in this version, so
`solve()` either finds a complete schedule or raises `SchedulingError`. Let's see that happen with
a conference that's clearly too short.

In [3]:
try:
    Schedule(presentations, unavailability, session_times, n_days=5).solve()
except SchedulingError as error:
    print(f"SchedulingError: {error}")

SchedulingError: no schedule satisfies every constraint for the given n_days


## Solving with a realistic conference length

Giving the busiest moderator enough headroom (10 days × 8 sessions = 80 slots, against a load of
~53) is enough to find a complete schedule. `solve()` returns a `DataFrame` indexed by presentation
`id`, with `day` and `session` columns — that's the entire output; a presentation's room is
whichever room its `moderator` is chairing that day, so it doesn't need its own column.

In [4]:
n_days = 10

schedule = Schedule(presentations, unavailability, session_times, n_days).solve()

print(f"scheduled {len(schedule)} of {len(presentations.data)} presentations")
schedule.head()

scheduled 290 of 290 presentations


,day,session
id,,
tby46,5,2
fql33,1,7
dt5008,9,2
hkc953,10,8
cy306,1,6


## Checking the busiest moderator isn't double-booked

The whole point of the exercise: every presentation the busiest moderator chairs should land on a
distinct `(day, session)` slot, since they can only be in one room at a time.

In [5]:
busiest_moderator = moderator_load.index[0]
their_ids = presentations.data.index[presentations.data["moderator"] == busiest_moderator]
their_slots = schedule.loc[their_ids, ["day", "session"]]

print(f"{busiest_moderator} chairs {len(their_ids)} presentations")
print(f"distinct (day, session) slots used: {their_slots.drop_duplicates().shape[0]}")
assert their_slots.drop_duplicates().shape[0] == len(their_ids)

Kelly Garner chairs 53 presentations
distinct (day, session) slots used: 53


## Checking unavailability was respected

Let's pick someone from `unavailable.csv` who has a restriction, find every presentation they're
involved in (as a student, supervisor, or moderator), and confirm none of them were scheduled
during a slot they marked as unavailable.

In [6]:
restricted_person = unavailability.data["person"].iloc[0]
person_rules = unavailability.data[unavailability.data["person"] == restricted_person]
print(f"checking: {restricted_person}")
person_rules

checking: Lori Mcmillan


,person,day,session
0,Lori Mcmillan,4,<NA>


In [7]:
role_columns = ["student", "s1_name", "s2_name", "moderator"]
is_involved = (presentations.data[role_columns] == restricted_person).any(axis=1)
involved_ids = presentations.data.index[is_involved]

for presentation_id in involved_ids:
    day, session = schedule.loc[presentation_id, ["day", "session"]]
    assert not unavailability.is_unavailable(restricted_person, day, session)

print(f"{len(involved_ids)} presentation(s) involving {restricted_person}, no violations")

4 presentation(s) involving Lori Mcmillan, no violations


## Seeing parallel rooms in action

Rooms aren't a separate resource — any number of presentations can share a `(day, session)` slot
as long as they have different moderators. Let's look at the busiest slot and confirm every
presentation scheduled there has a distinct moderator (i.e. they're genuinely running in parallel,
in different rooms, not clashing).

In [8]:
slot_sizes = schedule.groupby(["day", "session"]).size().sort_values(ascending=False)
busiest_day, busiest_session = slot_sizes.index[0]
print(f"busiest slot: day {busiest_day}, session {busiest_session} — {slot_sizes.iloc[0]} presentations in parallel")

ids_in_slot = schedule.index[(schedule["day"] == busiest_day) & (schedule["session"] == busiest_session)]
moderators_in_slot = presentations.data.loc[ids_in_slot, "moderator"]

moderators_in_slot

busiest slot: day 2, session 2 — 6 presentations in parallel


id
yg103       Cindy James
kr6067    Luis Reynolds
yt8732    Amanda Castro
wzh146     Amanda Ramos
pf2131     Kelly Garner
jj373      James Lowery
Name: moderator, dtype: str

In [9]:
assert moderators_in_slot.is_unique
print("all moderators in this slot are distinct — genuinely parallel, not clashing")

all moderators in this slot are distinct — genuinely parallel, not clashing


## Wrap-up

- `Presentations`, `Unavailability`, and `SessionTimes` load and validate the three input CSVs.
- `Schedule(presentations, unavailability, session_times, n_days).solve()` returns a `day`/`session`
  schedule, or raises `SchedulingError` if `n_days` isn't enough.
- A person can never be double-booked across roles or presentations at the same slot, and
  unavailability rules are always honoured.
- Rooms fall out of the moderator assignment for free — no separate room resource to manage.

This version is feasibility-only: `solve()` doesn't try to minimize the number of days or rooms
used, it just finds *a* valid schedule (or tells you none exists). Picking `n_days` is currently up
to you, as this example showed by trying 5 days first.